# 05 — RAG Setup: FAISS Vector DB

Builds the FAISS index from the fact-check knowledge base and demonstrates semantic retrieval.

**Model used:** `all-MiniLM-L6-v2` (sentence-transformers, ~80 MB, runs locally, free)

**Architecture:**
```
Query text
   ↓  encode (all-MiniLM-L6-v2)
Query embedding (384-dim)
   ↓  cosine similarity search
FAISS IndexFlatIP
   ↓
Top-K knowledge base documents
```

## 1. Install Dependencies

In [ ]:
# Run once
# !pip install faiss-cpu sentence-transformers

## 2. Import & Setup

In [ ]:
import sys
sys.path.insert(0, '../backend')

from agent.retriever import KNOWLEDGE_BASE, build_index, retrieve
import json

print(f'Knowledge base size: {len(KNOWLEDGE_BASE)} documents')
print('\nCategories:', list(set(d['category'] for d in KNOWLEDGE_BASE)))

## 3. Inspect Knowledge Base

In [ ]:
import pandas as pd

df_kb = pd.DataFrame(KNOWLEDGE_BASE)[['id', 'title', 'source', 'category']]
df_kb

## 4. Build FAISS Index

In [ ]:
success = build_index()
print('Index built successfully:', success)

## 5. Inspect Index

In [ ]:
import pickle, os

index_path = '../backend/agent/faiss_index.pkl'
if os.path.exists(index_path):
    with open(index_path, 'rb') as f:
        data = pickle.load(f)
    index = data['index']
    print(f'Index type: {type(index).__name__}')
    print(f'Total vectors: {index.ntotal}')
    print(f'Embedding dimension: {index.d}')
else:
    print('Index not found — run build_index() first or install faiss-cpu + sentence-transformers')

## 6. Retrieval Demo — Fake News Query

In [ ]:
query = 'BREAKING: shocking conspiracy cover-up they dont want you to know'
results = retrieve(query, top_k=3)

print(f'Query: "{query}"\n')
for i, doc in enumerate(results, 1):
    print(f'--- Result {i} (relevance: {doc.get("relevance_score", "N/A")}) ---')
    print(f'Title:   {doc["title"]}')
    print(f'Source:  {doc["source"]}')
    print(f'Content: {doc["content"][:150]}...')
    print()

## 7. Retrieval Demo — Real News Query

In [ ]:
query = 'reuters confirmed officials said according to study research data'
results = retrieve(query, top_k=3)

print(f'Query: "{query}"\n')
for i, doc in enumerate(results, 1):
    print(f'--- Result {i} (relevance: {doc.get("relevance_score", "N/A")}) ---')
    print(f'Title:   {doc["title"]}')
    print(f'Source:  {doc["source"]}')
    print()

## 8. Embedding Visualization (PCA)

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.decomposition import PCA
    import matplotlib.pyplot as plt
    import numpy as np

    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    texts = [d['title'] for d in KNOWLEDGE_BASE]
    categories = [d['category'] for d in KNOWLEDGE_BASE]
    embeddings = encoder.encode(texts, convert_to_numpy=True)

    pca = PCA(n_components=2)
    coords = pca.fit_transform(embeddings)

    unique_cats = list(set(categories))
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_cats)))
    cat_color = {c: colors[i] for i, c in enumerate(unique_cats)}

    plt.figure(figsize=(10, 6))
    for i, (x, y) in enumerate(coords):
        plt.scatter(x, y, color=cat_color[categories[i]], s=80)
        plt.annotate(texts[i][:30], (x, y), fontsize=7, ha='center', va='bottom')

    handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=cat_color[c], label=c) for c in unique_cats]
    plt.legend(handles=handles, loc='best', fontsize=8)
    plt.title('Knowledge Base Embeddings (PCA 2D)')
    plt.tight_layout()
    plt.show()
    print(f'Explained variance: {pca.explained_variance_ratio_.sum():.1%}')
except ImportError:
    print('sentence-transformers not installed — skipping visualization')

## Summary

| Component | Detail |
|---|---|
| Encoder | `all-MiniLM-L6-v2` (384-dim) |
| Index type | `faiss.IndexFlatIP` (cosine via L2-normalized vectors) |
| KB size | 10 curated fact-check documents |
| Fallback | Keyword overlap if FAISS unavailable |
| Retrieval | Top-3 by cosine similarity |